<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-01-logica/07%20-%20Validade%20e%20Inferencia%20Logica%20na%20Manufatura%20Flexivel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 07 - Notebook: Validade de Argumentos e Inferência Lógica em Sistemas de Manufatura Flexível

Neste notebook implementamos a classe **`ValidadorLogicoManufaturaFlexivel`** para verificar formalmente a validade de regras de classificação e roteamento de peças em uma célula de manufatura flexível. Exploramos tabela-verdade, prova por refutação e detecção de falácias lógicas aplicadas à automação industrial.


In [3]:

def formatar_tabela(dados):
    if not dados:
        return "Tabela Vazia"
    colunas=list(dados[0].keys())
    larguras={c:len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c]=max(larguras[c],len(str(row.get(c,""))))
    header=" | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor="-+-".join("-"*larguras[c] for c in colunas)
    linhas=[header,divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c,'')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

import itertools
from typing import List, Dict, Callable, Any

class ValidadorLogicoManufaturaFlexivel:

    @staticmethod
    def verificar_argumento_tabela_verdade(variaveis,premissas,conclusao):
        total_estados=2**len(variaveis)
        linhas_criticas=0
        linhas_validas=0
        contraexemplos=[]

        for combo in itertools.product([False,True], repeat=len(variaveis)):
            env=dict(zip(variaveis,combo))

            if all(p(env) for p in premissas):
                linhas_criticas+=1
                if conclusao(env):
                    linhas_validas+=1
                else:
                    contraexemplos.append(env)

        valido=(linhas_criticas>0) and (linhas_criticas==linhas_validas)

        return {
            "Total Estados (2^n)": total_estados,
            "Estados com Premissas True": linhas_criticas,
            "Estados com Conclusão True": linhas_validas,
            "Válido": valido,
            "Resultado Semântico":"REGRA VÁLIDA DE MANUFATURA" if valido else "REGRA INVÁLIDA",
            "Contraexemplos": contraexemplos
        }

    @staticmethod
    def verificar_por_refutacao(variaveis,premissas,conclusao):
        modelos=[]
        for combo in itertools.product([False,True], repeat=len(variaveis)):
            env=dict(zip(variaveis,combo))
            if all(p(env) for p in premissas) and not conclusao(env):
                modelos.append(env)

        return {
            "Refutação Bem-Sucedida": len(modelos)==0,
            "Conclusão":"ARGUMENTO VÁLIDO" if len(modelos)==0 else "ARGUMENTO INVÁLIDO"
        }

print("[OK] ValidadorLogicoManufaturaFlexivel inicializado!")


[OK] ValidadorLogicoManufaturaFlexivel inicializado!


In [4]:

# TESTES DA CÉLULA DE MANUFATURA FLEXÍVEL

# 1. Modus Ponens
# Se peça vermelha -> Esteira A
vars_mp=['c1','A1']
p1=lambda e:(not e['c1']) or e['A1']
p2=lambda e:e['c1']
c=lambda e:e['A1']
res_mp=ValidadorLogicoManufaturaFlexivel.verificar_argumento_tabela_verdade(vars_mp,[p1,p2],c)

# 2. Modus Tollens
# Se passou no sensor de saída -> contador atualizado
vars_mt=['saida','contador']
p1=lambda e:(not e['saida']) or e['contador']
p2=lambda e:not e['contador']
c=lambda e:not e['saida']
res_mt=ValidadorLogicoManufaturaFlexivel.verificar_argumento_tabela_verdade(vars_mt,[p1,p2],c)

# 3. Silogismo Hipotético
# metálica -> inspeção ; inspeção -> etiqueta
vars_sh=['m1','INS','ETQ']
p1=lambda e:(not e['m1']) or e['INS']
p2=lambda e:(not e['INS']) or e['ETQ']
c=lambda e:(not e['m1']) or e['ETQ']
res_sh=ValidadorLogicoManufaturaFlexivel.verificar_argumento_tabela_verdade(vars_sh,[p1,p2],c)

# 4. Resolução
vars_res=['c1','c2','A1']
p1=lambda e:e['c1'] or e['c2']
p2=lambda e:(not e['c1']) or e['A1']
c=lambda e:e['c2'] or e['A1']
res_res=ValidadorLogicoManufaturaFlexivel.verificar_argumento_tabela_verdade(vars_res,[p1,p2],c)

# 5. Falácia
vars_fal=['c1','A1']
p1=lambda e:(not e['c1']) or e['A1']
p2=lambda e:e['A1']
c=lambda e:e['c1']
res_fal=ValidadorLogicoManufaturaFlexivel.verificar_argumento_tabela_verdade(vars_fal,[p1,p2],c)

relatorio=[
{"Esquema":"Modus Ponens","Variáveis":"c1,A1","Válido":res_mp["Válido"]},
{"Esquema":"Modus Tollens","Variáveis":"saida,contador","Válido":res_mt["Válido"]},
{"Esquema":"Silogismo Hipotético","Variáveis":"m1,INS,ETQ","Válido":res_sh["Válido"]},
{"Esquema":"Resolução","Variáveis":"c1,c2,A1","Válido":res_res["Válido"]},
{"Esquema":"Afirmação do Consequente","Variáveis":"c1,A1","Válido":res_fal["Válido"]}
]

print("=== RELATÓRIO DE VALIDAÇÃO LÓGICA DA MANUFATURA FLEXÍVEL ===")
print(formatar_tabela(relatorio))

assert res_mp["Válido"] is True
assert res_mt["Válido"] is True
assert res_sh["Válido"] is True
assert res_res["Válido"] is True
assert res_fal["Válido"] is False

print("\n[OK] Todos os testes executados com sucesso!")


=== RELATÓRIO DE VALIDAÇÃO LÓGICA DA MANUFATURA FLEXÍVEL ===
Esquema                  | Variáveis      | Válido
-------------------------+----------------+-------
Modus Ponens             | c1,A1          | True  
Modus Tollens            | saida,contador | True  
Silogismo Hipotético     | m1,INS,ETQ     | True  
Resolução                | c1,c2,A1       | True  
Afirmação do Consequente | c1,A1          | False 

[OK] Todos os testes executados com sucesso!
